# DataFrame API Cheatsheet — Databricks Spark Associate Exam

**Exam Domain: Developing DataFrame/DataSet API Applications (30%)**

## READING DATA

In [ ]:
# CSV — text, requires options
df = spark.read.option("header", True).option("inferSchema", True).csv("path")
df = spark.read.schema(my_schema).csv("path")  # explicit schema (faster)

# JSON — text, nested, schema inferred or explicit
df = spark.read.json("path")
df = spark.read.schema(my_schema).json("path")

# Parquet — binary, columnar, schema embedded (no options needed)
df = spark.read.parquet("path")

# Delta — Parquet + transaction log, ACID, versioned
df = spark.read.format("delta").load("path")
df = spark.table("catalog.schema.table")  # managed table

## WRITING DATA

In [ ]:
# Write modes: "error" (default), "overwrite", "append", "ignore"
df.write.mode("overwrite").parquet("path")
df.write.mode("append").saveAsTable("my_table")  # managed Delta table
df.write.partitionBy("year", "month").parquet("path")  # partition on write

## COLUMN SELECTION

In [ ]:
from pyspark.sql.functions import col

df.select("col1", "col2")               # string names
df.select(col("col1"), col("col2"))      # col() objects
df.select(df.col1, df.col2)             # dot notation
df.select(df["col1"], df["col2"])       # bracket notation

# Drop columns
df.drop("col1", "col2")

## COLUMN OPERATIONS

In [ ]:
# Add or replace a column
df.withColumn("new_col", col("x") + col("y"))

# Rename a column
df.withColumnRenamed("old_name", "new_name")

# Alias an expression
col("total_amount").alias("fare")

# Cast to a different type
col("passenger_count").cast("integer")
col("amount").cast(DoubleType())

## FILTERING

In [ ]:
# filter() and where() are IDENTICAL
df.filter(col("x") > 5)
df.where(col("x") > 5)

# Multiple conditions — use & (AND), | (OR), ~ (NOT), with parentheses
df.filter((col("x") > 5) & (col("y") < 10))
df.filter((col("x") > 5) | (col("y") < 10))

# Membership and range
df.filter(col("x").isin(1, 2, 3))
df.filter(col("x").between(5, 15))
df.filter(col("x").isNull())
df.filter(col("x").isNotNull())
df.filter(col("name").like("%spark%"))

## TRANSFORMATIONS

In [ ]:
df.distinct()                                # unique rows
df.dropDuplicates(["col1", "col2"])          # unique by specific columns
df.sort(col("x").desc())                     # sort descending
df.orderBy(col("x").asc(), col("y").desc())  # multi-column sort
df.limit(100)                                # first N rows

## ACTIONS (trigger execution)

In [ ]:
df.show(5)           # prints 5 rows (default 20)
df.count()           # total row count
df.collect()         # ALL rows to driver — OOM risk on large data!
df.take(5)           # first 5 rows to driver
df.first()           # first row
df.head(5)           # same as take(5)
df.toPandas()        # ALL data to driver as pandas DF — OOM risk!

## AGGREGATIONS

In [ ]:
from pyspark.sql.functions import count, sum, avg, min, max, round

df.groupBy("col").count()
df.groupBy("col").agg(
    count("*").alias("cnt"),
    round(avg("amount"), 2).alias("avg_amt"),
    sum("amount").alias("total"),
    min("amount").alias("min_amt"),
    max("amount").alias("max_amt"),
)

## JOINS

In [ ]:
# Join types: "inner" (default), "left", "right", "full", "left_semi", "left_anti", "cross"
df1.join(df2, on="key", how="inner")                        # same column name
df1.join(df2, df1.key == df2.id, how="left")                # different names
df1.join(broadcast(df2), on="key")                          # broadcast small table

# Semi join — filter left by match in right (no right columns added)
# Anti join — filter left by NO match in right

## SET OPERATIONS

In [ ]:
df1.union(df2)         # all rows (by position), keeps duplicates
df1.unionByName(df2)   # all rows (by name), keeps duplicates
df1.intersect(df2)     # rows in both
df1.exceptAll(df2)     # rows in df1 not in df2

## SCHEMA

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=False),
])
df = spark.read.schema(schema).json("path")
df.printSchema()